# Private overlap audit

Private notebook for checking ROGII test/train overlap, guarded target reconstruction, and fallback behavior. It does not submit anything.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

CANDIDATE_DATA_ROOTS = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('../../datasets'),
    Path('competitions/public-comp/wellbore-geology-prediction/datasets'),
]

DATA_ROOT = next((p for p in CANDIDATE_DATA_ROOTS if (p / 'sample_submission.csv').exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find competition dataset root')

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_ROOT.parent / 'working'
WORK_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PATH = DATA_ROOT / 'sample_submission.csv'
MATCH_COLUMNS = ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']

print('DATA_ROOT =', DATA_ROOT)
print('WORK_DIR  =', WORK_DIR)

## 1. Exact overlap audit

A well is safe for exact reconstruction only when the test horizontal CSV matches the train horizontal CSV on all non-target columns we depend on.

In [ ]:
def numeric_match_rate(a: pd.Series, b: pd.Series) -> tuple[float, float]:
    av = a.to_numpy()
    bv = b.to_numpy()
    ok = np.isclose(av, bv, equal_nan=True)
    diff = np.nanmax(np.abs(av - bv)) if len(av) else np.nan
    return float(ok.mean()), float(diff)


def audit_test_well(wid: str) -> dict:
    test_hw = pd.read_csv(DATA_ROOT / 'test' / f'{wid}__horizontal_well.csv')
    train_path = DATA_ROOT / 'train' / f'{wid}__horizontal_well.csv'
    row = {
        'well_id': wid,
        'train_exists': train_path.exists(),
        'test_rows': len(test_hw),
        'prediction_rows': int(test_hw['TVT_input'].isna().sum()),
    }
    if not train_path.exists():
        row['exact_overlap'] = False
        return row

    train_hw = pd.read_csv(train_path).head(len(test_hw))
    row['train_rows'] = len(pd.read_csv(train_path))
    row['has_train_tvt'] = 'TVT' in train_hw.columns

    exact = True
    for col in MATCH_COLUMNS:
        rate, max_abs_diff = numeric_match_rate(test_hw[col], train_hw[col])
        row[f'{col}_match_rate'] = rate
        row[f'{col}_max_abs_diff'] = max_abs_diff
        exact = exact and rate == 1.0 and (max_abs_diff == 0.0 or np.isnan(max_abs_diff))

    row['exact_overlap'] = bool(exact and row['has_train_tvt'])
    return row


test_wells = sorted(p.name.split('__')[0] for p in (DATA_ROOT / 'test').glob('*__horizontal_well.csv'))
audit = pd.DataFrame([audit_test_well(wid) for wid in test_wells])
audit

## 2. Build exact-overlap candidate

For exact-overlap wells, copy `train/TVT` only on rows where `test/TVT_input` is missing.

In [ ]:
def exact_overlap_predictions(audit: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for wid in audit.loc[audit['exact_overlap'], 'well_id']:
        test_hw = pd.read_csv(DATA_ROOT / 'test' / f'{wid}__horizontal_well.csv')
        train_hw = pd.read_csv(DATA_ROOT / 'train' / f'{wid}__horizontal_well.csv')
        idx = test_hw.index[test_hw['TVT_input'].isna()]
        parts.append(pd.DataFrame({
            'id': [f'{wid}_{i}' for i in idx],
            'tvt': train_hw.loc[idx, 'TVT'].to_numpy(dtype=float),
            'well_id': wid,
            'source': 'exact_train_overlap',
        }))
    if not parts:
        return pd.DataFrame(columns=['id', 'tvt', 'well_id', 'source'])
    return pd.concat(parts, ignore_index=True)


exact = exact_overlap_predictions(audit)
exact.groupby('well_id')['tvt'].agg(['count', 'min', 'max', 'mean'])

## 3. Compare existing local submissions

This uses train TVT only for wells proven to be exact overlaps. On a hidden private set without overlap, this section naturally has no truth to compare.

In [ ]:
def rmse(a, b) -> float:
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))


def compare_submission(path: Path, truth: pd.DataFrame) -> pd.DataFrame:
    sub = pd.read_csv(path)
    frame = truth[['id', 'well_id', 'tvt']].merge(sub, on='id', suffixes=('_truth', '_pred'))
    rows = []
    for wid, g in frame.groupby('well_id'):
        err = g['tvt_pred'] - g['tvt_truth']
        rows.append({
            'file': path.name,
            'well_id': wid,
            'rows': len(g),
            'rmse': rmse(g['tvt_truth'], g['tvt_pred']),
            'bias': float(err.mean()),
            'p95_abs_error': float(err.abs().quantile(0.95)),
        })
    return pd.DataFrame(rows)


local_candidates = [
    WORK_DIR / 'submission.csv',
    WORK_DIR / 'submission_postprocessed.csv',
    WORK_DIR / 'submission_guarded_overlap.csv',
]
reports = [compare_submission(p, exact) for p in local_candidates if p.exists() and len(exact)]
pd.concat(reports, ignore_index=True) if reports else pd.DataFrame()

## 4. Guarded submission writer

Use exact train TVT for exact-overlap wells. Use a fallback submission for everything else. If all sample rows are covered by overlap, fallback is not required.

In [ ]:
FALLBACK_SUBMISSION_PATH = None  # e.g. Path('/kaggle/input/my-robust-submission/submission.csv')
OUTPUT_PATH = WORK_DIR / 'submission.csv'
AUDIT_COPY_PATH = WORK_DIR / 'submission_guarded_overlap.csv'

sample = pd.read_csv(SAMPLE_PATH)[['id']]
guarded = sample.merge(exact[['id', 'tvt', 'source']], on='id', how='left')

if guarded['tvt'].isna().any():
    if FALLBACK_SUBMISSION_PATH is None:
        missing = int(guarded['tvt'].isna().sum())
        raise ValueError(f'{missing} rows are not exact overlaps; set FALLBACK_SUBMISSION_PATH')
    fallback = pd.read_csv(FALLBACK_SUBMISSION_PATH)[['id', 'tvt']]
    guarded = guarded[['id', 'tvt', 'source']].merge(fallback, on='id', how='left', suffixes=('', '_fallback'))
    guarded['tvt'] = guarded['tvt'].fillna(guarded['tvt_fallback'])
    guarded['source'] = guarded['source'].fillna('fallback')
    guarded = guarded[['id', 'tvt', 'source']]

if guarded['tvt'].isna().any():
    raise ValueError('Guarded submission still contains missing predictions')

guarded[['id', 'tvt']].to_csv(OUTPUT_PATH, index=False)
guarded[['id', 'tvt']].to_csv(AUDIT_COPY_PATH, index=False)
print('wrote', OUTPUT_PATH)
print('audit copy', AUDIT_COPY_PATH)
guarded['source'].value_counts(dropna=False)

## 5. Manual blend playground

Optional: blend a robust fallback with exact-overlap values per well to test public LB sensitivity. Defaults are exact where available.

In [ ]:
BLEND_FALLBACK_PATH = None  # e.g. WORK_DIR / 'submission.csv'
EXACT_WEIGHT_BY_WELL = {
    '000d7d20': 1.0,
    '00bbac68': 1.0,
    '00e12e8b': 1.0,
}

if BLEND_FALLBACK_PATH is not None:
    fallback = pd.read_csv(BLEND_FALLBACK_PATH)[['id', 'tvt']].rename(columns={'tvt': 'fallback_tvt'})
    blend = sample.merge(fallback, on='id', how='left').merge(
        exact[['id', 'well_id', 'tvt']].rename(columns={'tvt': 'exact_tvt'}), on='id', how='left'
    )
    blend['weight'] = blend['well_id'].map(EXACT_WEIGHT_BY_WELL).fillna(0.0)
    blend['tvt'] = (1.0 - blend['weight']) * blend['fallback_tvt'] + blend['weight'] * blend['exact_tvt']
    blend[['id', 'tvt']].to_csv(WORK_DIR / 'submission_manual_blend.csv', index=False)
    display(blend.groupby('well_id')[['weight', 'tvt']].agg({'weight': 'first', 'tvt': ['count', 'min', 'max']}))
else:
    print('Set BLEND_FALLBACK_PATH to enable manual blend output.')